In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load model + tokenizer
model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name,
    torch_dtype=torch.float16, device_map="auto")

# 2. Register a forward hook on the lm_head (linear projection)
linear_outputs = []

def hook_fn(module, inputs, output):
    # output shape: (batch_size, seq_len, vocab_size)
    linear_outputs.append(output.detach().cpu())

model.lm_head.register_forward_hook(hook_fn)


In [ ]:
import torch.nn.functional as F
prompt = "Hello world"
input1 = tokenizer(prompt, return_tensors="pt").to(model.device)

# First pass
logits1 = linear_outputs[-1]  # shape (1, seq_len, vocab_size)
print(logits1.shape)

# Copy to freeze it
logits1_copy = logits1.clone()

# Clear hook outputs
linear_outputs.clear()

# Second pass 
prompt2 = "Goodbye world"
input2 = tokenizer(prompt2, return_tensors="pt").to(model.device)
_ = model(**input2)
logits2 = linear_outputs[-1]

# Convert logits to probabilities
p1 = F.softmax(logits1_copy.float(), dim=-1)
p2 = F.softmax(logits2.float(), dim=-1)

# KL divergence: average per token, sum over vocab
kl = F.kl_div(p1, p2, reduction="batchmean")  # computes KL(p1 || p2) 

print("KL divergence:", kl.item())

torch.Size([1, 3, 151936])
KL divergence: -16.34030532836914
